# Assignment 9 — Multi-class Classification

**Book/data context:** Three two-feature classification datasets (`dataset_1.csv`, `dataset_2.csv`, `dataset_3.csv`).

This notebook explores the datasets, creates one stratified 80/20 split per dataset, fits all 15 requested classifier configurations, computes train/test metrics, and saves decision-boundary and micro-ROC plots. The random seed is fixed at 42. Run from top to bottom with the three CSV files in the same directory.

## 1. Data exploration and common splits

Each dataset has columns `x`, `y`, and `cluster`. The same split is reused by all classifiers within a dataset. `stratify=y` preserves class proportions in both partitions.

In [1]:
# Build the full assignment workflow, execute it, and package the requested deliverables.
import os, json, textwrap, zipfile, warnings, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             roc_auc_score, roc_curve, auc)
from sklearn.exceptions import ConvergenceWarning
from IPython.display import display, Image as IPyImage

OUT = Path("assignment_9_work")
OUT.mkdir(exist_ok=True)
DATA_PATHS = [Path("dataset_1.csv"), Path("dataset_2.csv"), Path("dataset_3.csv")]
DATA_NAMES = [p.stem for p in DATA_PATHS]

# Validate schemas and explore data.
datasets = {}
for name, path in zip(DATA_NAMES, DATA_PATHS):
    df = pd.read_csv(path)
    if not {"x", "y", "cluster"}.issubset(df.columns):
        raise ValueError(f"{path.name} must contain x, y, and cluster columns.")
    if len(df) != 1500:
        warnings.warn(f"{path.name} has {len(df)} rows rather than the expected 1,500.")
    if df[["x", "y", "cluster"]].isna().any().any():
        raise ValueError(f"{path.name} contains missing values.")
    datasets[name] = df[["x", "y", "cluster"]].copy()

class_counts = {name: df["cluster"].value_counts().sort_index() for name, df in datasets.items()}
print("DATASET PREVIEW AND CLASS COUNTS")
for name, df in datasets.items():
    print(f"\n{name}:")
    display(df.head())
    display(class_counts[name].rename("count").to_frame())

# Common model specifications: 2 + 2 + 4 + 3 + 4 = 15.
def make_models():
    models = []
    models.append(("Logistic Regression (original)", LogisticRegression(max_iter=2000, random_state=42), "original"))
    models.append(("Logistic Regression (degree-3)", LogisticRegression(max_iter=2000, random_state=42), "poly"))
    models.append(("SVC (linear, C=0.5)", SVC(kernel="linear", C=0.5, probability=True, random_state=42), "original"))
    models.append(("SVC (RBF, C=2, gamma=scale)", SVC(kernel="rbf", C=2, gamma="scale", probability=True, random_state=42), "original"))
    for leaf in (2, 8):
        for depth in (3, 6):
            models.append((f"Random Forest (leaf={leaf}, depth={depth})",
                           RandomForestClassifier(n_estimators=100, min_samples_leaf=leaf,
                                                   max_depth=depth, random_state=42), "original"))
    for k in (1, 5, 11):
        models.append((f"k-NN (k={k})", KNeighborsClassifier(n_neighbors=k), "original"))
    for layers in ((2,), (6,), (6, 3), (3, 6, 3)):
        models.append((f"MLP {layers}", MLPClassifier(hidden_layer_sizes=layers,
                                                       max_iter=1500, random_state=42), "original"))
    assert len(models) == 15
    return models

# Utility functions for aligned class probabilities and metrics.
def aligned_probabilities(model, X):
    raw = model.predict_proba(X)
    aligned = np.zeros((len(X), len(model.classes_)), dtype=float)
    # predict_proba columns are in model.classes_ order; preserve that order explicitly.
    aligned[:] = raw
    return aligned, np.asarray(model.classes_)

def compute_metrics(y_true, y_pred, prob, prob_classes, all_classes):
    result = {"accuracy": accuracy_score(y_true, y_pred)}
    p, r, f, support = precision_recall_fscore_support(
        y_true, y_pred, labels=all_classes, zero_division=0
    )
    present = [j for j, c in enumerate(all_classes) if np.any(np.asarray(y_true) == c)]
    for j, c in enumerate(all_classes):
        prefix = f"class_{c}"
        if j in present:
            result[f"precision_{prefix}"] = p[j]
            result[f"recall_{prefix}"] = r[j]
            result[f"f1_{prefix}"] = f[j]
        else:
            result[f"precision_{prefix}"] = np.nan
            result[f"recall_{prefix}"] = np.nan
            result[f"f1_{prefix}"] = np.nan

    y_arr = np.asarray(y_true)
    aucs = []
    for c in all_classes:
        if np.any(y_arr == c) and np.any(y_arr != c) and c in prob_classes:
            col = int(np.where(prob_classes == c)[0][0])
            auc_value = roc_auc_score((y_arr == c).astype(int), prob[:, col])
            result[f"auc_class_{c}"] = auc_value
            aucs.append(auc_value)
        else:
            result[f"auc_class_{c}"] = np.nan
    result["macro_precision"] = float(np.mean([p[j] for j in present])) if present else np.nan
    result["macro_recall"] = float(np.mean([r[j] for j in present])) if present else np.nan
    result["macro_f1"] = float(np.mean([f[j] for j in present])) if present else np.nan
    result["macro_auc"] = float(np.mean(aucs)) if aucs else np.nan
    return result

def make_grid(X, points=240, pad=0.45):
    x_min, x_max = X[:, 0].min() - pad, X[:, 0].max() + pad
    y_min, y_max = X[:, 1].min() - pad, X[:, 1].max() + pad
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, points),
                         np.linspace(y_min, y_max, points))
    return xx, yy, np.c_[xx.ravel(), yy.ravel()]

def plot_model_output(dataset_name, model_name, model, feature_mode, poly, X_train, X_test,
                      y_train, y_test, classes, model_dir):
    xx, yy, grid = make_grid(np.vstack([X_train, X_test]))
    grid_features = poly.transform(grid) if feature_mode == "poly" else grid
    pred_grid = model.predict(grid_features).reshape(xx.shape)
    X_test_features = poly.transform(X_test) if feature_mode == "poly" else X_test
    prob_test, prob_classes = aligned_probabilities(model, X_test_features)
    X_train_features = poly.transform(X_train) if feature_mode == "poly" else X_train
    prob_train, train_prob_classes = aligned_probabilities(model, X_train_features)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
    ax = axes[0]
    ax.contourf(xx, yy, pred_grid, alpha=0.22, levels=len(classes)+1)
    for c in classes:
        mask = (y_test == c)
        ax.scatter(X_test[mask, 0], X_test[mask, 1], s=13, alpha=0.78, label=f"Class {c}")
    ax.set_title("Decision regions + test observations")
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.legend(title="True class", fontsize=7, loc="best")

    ax = axes[1]
    # Flatten one-hot indicators and the matching class probability columns for micro ROC.
    y_train_bin = label_binarize(y_train, classes=classes)
    y_test_bin = label_binarize(y_test, classes=classes)
    train_order = [int(np.where(train_prob_classes == c)[0][0]) for c in classes]
    test_order = [int(np.where(prob_classes == c)[0][0]) for c in classes]
    train_scores = prob_train[:, train_order]
    test_scores = prob_test[:, test_order]
    fpr_tr, tpr_tr, _ = roc_curve(y_train_bin.ravel(), train_scores.ravel())
    fpr_te, tpr_te, _ = roc_curve(y_test_bin.ravel(), test_scores.ravel())
    auc_tr = auc(fpr_tr, tpr_tr)
    auc_te = auc(fpr_te, tpr_te)
    ax.plot(fpr_tr, tpr_tr, label=f"Train micro AUC = {auc_tr:.3f}")
    ax.plot(fpr_te, tpr_te, label=f"Test micro AUC = {auc_te:.3f}")
    ax.plot([0, 1], [0, 1], linestyle="--", label="Chance")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_title("One-vs-rest micro ROC")
    ax.legend(fontsize=8, loc="lower right")
    fig.suptitle(f"{dataset_name} — {model_name}", fontsize=12)
    fig.tight_layout()
    safe = "".join(ch if ch.isalnum() else "_" for ch in model_name).strip("_")
    fig_path = model_dir / f"{dataset_name}_{safe}.png"
    fig.savefig(fig_path, dpi=115, bbox_inches="tight")
    return fig, fig_path, auc_tr, auc_te

all_rows = []
all_plot_records = []
model_fits = {}
convergence_notes = []
exploration_plot_paths = []

# Initial scatter plots for each dataset.
for name, df in datasets.items():
    fig, ax = plt.subplots(figsize=(6.5, 5))
    for c in sorted(df.cluster.unique()):
        m = df.cluster == c
        ax.scatter(df.loc[m, "x"], df.loc[m, "y"], s=13, alpha=.75, label=f"Class {c}")
    ax.set(title=f"{name}: feature space", xlabel="x", ylabel="y")
    ax.legend()
    fig.tight_layout()
    p = OUT / f"{name}_exploration.png"
    fig.savefig(p, dpi=130)
    exploration_plot_paths.append(p)
    plt.close(fig)

for dataset_name, df in datasets.items():
    X = df[["x", "y"]].to_numpy(dtype=float)
    y = df["cluster"].to_numpy()
    classes = np.sort(np.unique(y))
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
    # Fit polynomial transformer on training data only; omit constant bias feature.
    poly = PolynomialFeatures(degree=3, include_bias=False)
    poly.fit(X_train)
    model_dir = OUT / dataset_name
    model_dir.mkdir(exist_ok=True)

    for model_name, model, feature_mode in make_models():
        Xtr = poly.transform(X_train) if feature_mode == "poly" else X_train
        Xte = poly.transform(X_test) if feature_mode == "poly" else X_test
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model.fit(Xtr, y_train)
        for w in caught:
            if issubclass(w.category, ConvergenceWarning):
                convergence_notes.append({"dataset": dataset_name, "model": model_name, "warning": str(w.message)})
        model_fits[(dataset_name, model_name)] = (model, feature_mode, poly, X_train, X_test, y_train, y_test, classes)

        for split, XX, yy_true in (("train", Xtr, y_train), ("test", Xte, y_test)):
            y_pred = model.predict(XX)
            probs, prob_classes = aligned_probabilities(model, XX)
            metrics = compute_metrics(yy_true, y_pred, probs, prob_classes, classes)
            all_rows.append({"dataset": dataset_name, "model": model_name, "split": split, **metrics})

        fig, fig_path, micro_train_auc, micro_test_auc = plot_model_output(
            dataset_name, model_name, model, feature_mode, poly,
            X_train, X_test, y_train, y_test, classes, model_dir
        )
        all_plot_records.append({
            "dataset": dataset_name, "model": model_name, "path": fig_path,
            "micro_train_auc": micro_train_auc, "micro_test_auc": micro_test_auc
        })
        plt.close(fig)

results = pd.DataFrame(all_rows)
results_path = OUT / "yourname_assignment_9_results.csv"
results.to_csv(results_path, index=False, na_rep="")
print("Completed model fitting:", len(model_fits), "models;", len(results), "metric rows.")
print("Convergence warnings:", len(convergence_notes))

DATASET PREVIEW AND CLASS COUNTS

dataset_1:


,x,y,cluster
0,1.204876,-1.877549,3
1,0.190524,0.682476,2
2,-2.524391,-3.200064,1
3,-1.401345,0.713046,2
4,-0.527609,1.229975,2


,count
cluster,
1,375
2,375
3,375
4,375



dataset_2:


,x,y,cluster
0,-2.911114,-0.214988,4
1,-1.513595,-0.187763,2
2,0.458368,-0.632647,1
3,-2.518231,0.165471,3
4,-0.414653,-1.429693,2


,count
cluster,
1,300
2,300
3,300
4,300
5,300



dataset_3:


,x,y,cluster
0,0.093224,1.074345,5
1,1.502937,-1.995782,2
2,-1.701505,-1.055810,1
3,0.667784,-1.919564,3
4,-0.262354,-1.924619,2


,count
cluster,
1,250
2,250
3,250
4,250
5,250
6,250


Completed model fitting: 45 models; 90 metric rows.
Convergence warnings: 1


## 2. Polynomial features, kernels, and model interpretation

For two inputs \(x_1,x_2\), the degree-3 polynomial expansion without a bias term produces:

- Degree 1: \(x_1, x_2\)
- Degree 2: \(x_1^2, x_1x_2, x_2^2\)
- Degree 3: \(x_1^3, x_1^2x_2, x_1x_2^2, x_2^3\)

The linear SVC kernel learns a straight hyperplane in the original feature space. The RBF kernel uses similarity that decreases with distance, allowing curved and more flexible boundaries. Polynomial features are fitted on training data and the same fitted transformer is applied to test data and the plotting grid.

## 3. Evaluation

The CSV contains one row per dataset/model/split (3 × 15 × 2 = 90). Per-class metrics are computed in a one-versus-rest manner. Macro metrics are unweighted means over classes. Probability columns are kept in the estimator's `classes_` order and explicitly aligned to the dataset class order before ROC calculations. Missing-class metric cells are left blank; undefined precision/recall/F1 use `zero_division=0`.

In [2]:
# Compact test-set leaderboard and train-test gaps.
summary = results.pivot_table(index=["dataset", "model"], columns="split",
                              values=["accuracy", "macro_f1", "macro_auc"])
summary.columns = [f"{metric}_{split}" for metric, split in summary.columns]
summary = summary.reset_index()
summary["macro_f1_gap"] = summary["macro_f1_train"] - summary["macro_f1_test"]
summary["accuracy_gap"] = summary["accuracy_train"] - summary["accuracy_test"]
for dataset in DATA_NAMES:
    print(dataset)
    display(summary[summary.dataset == dataset].sort_values(
        ["macro_f1_test", "accuracy_test"], ascending=False
    )[["model", "accuracy_test", "macro_f1_test", "macro_auc_test",
       "accuracy_gap", "macro_f1_gap"]].head(5).round(4))
# Copy the required common results CSV beside this notebook.
import shutil
shutil.copy2(results_path, Path("yourname_assignment_9_results.csv"))

dataset_1


,model,accuracy_test,macro_f1_test,macro_auc_test,accuracy_gap,macro_f1_gap
0,Logistic Regression (degree-3),0.9600,0.9600,0.9978,-0.0075,-0.0075
6,"Random Forest (leaf=2, depth=3)",0.9600,0.9600,0.9956,-0.0025,-0.0025
8,"Random Forest (leaf=8, depth=3)",0.9600,0.9600,0.9963,-0.0017,-0.0017
13,k-NN (k=11),0.9600,0.9600,0.9940,-0.0092,-0.0092
7,"Random Forest (leaf=2, depth=6)",0.9567,0.9567,0.9970,0.0117,0.0116


dataset_2


,model,accuracy_test,macro_f1_test,macro_auc_test,accuracy_gap,macro_f1_gap
28,k-NN (k=11),0.9767,0.9765,0.9982,0.0000,0.0001
25,"SVC (RBF, C=2, gamma=scale)",0.9700,0.9700,0.9988,0.0092,0.0092
29,k-NN (k=5),0.9700,0.9698,0.9969,0.0100,0.0101
27,k-NN (k=1),0.9633,0.9635,0.9771,0.0367,0.0365
15,Logistic Regression (degree-3),0.9633,0.9632,0.9989,0.0150,0.0151


dataset_3


,model,accuracy_test,macro_f1_test,macro_auc_test,accuracy_gap,macro_f1_gap
37,"Random Forest (leaf=2, depth=6)",0.8533,0.8542,0.9859,0.0275,0.0271
39,"Random Forest (leaf=8, depth=6)",0.8467,0.8479,0.9870,0.0050,0.0041
31,Logistic Regression (original),0.8433,0.8442,0.9794,-0.0225,-0.0233
33,"MLP (3, 6, 3)",0.8433,0.8442,0.9801,-0.0167,-0.0174
34,"MLP (6, 3)",0.8433,0.8439,0.9834,-0.0133,-0.0146


WindowsPath('yourname_assignment_9_results.csv')

## 4. Observations and conclusions

### Dataset 1
Degree-3 logistic regression, depth-3 random forests, and 11-NN are tied at 0.960 test accuracy and macro-F1. The original logistic model is also strong (0.957), so a linear separator is already a good approximation; polynomial terms offer only a small improvement. k-NN with k=1 has a larger train–test macro-F1 gap (0.070), consistent with a highly local, variance-prone boundary.

### Dataset 2
The data strongly reward nonlinear decision functions. Original logistic regression and linear SVC obtain only 0.220 and 0.303 test accuracy, while degree-3 logistic regression reaches 0.963, RBF SVC 0.970, and 11-NN 0.977. The degree-3 expansion captures useful curved structure; the RBF kernel and neighborhood averaging are also effective. Forests with depth 3 underfit (about 0.70–0.73 accuracy), whereas depth 6 improves to about 0.91–0.92. The one-hidden-layer MLP with six units reaches 0.803, and its run hits the 1,500-iteration limit without convergence; deeper small networks perform better.

### Dataset 3
This is the hardest dataset: the best test result is depth-6 random forest with leaf size 2 (0.853 accuracy, 0.854 macro-F1), followed closely by several MLP and logistic/SVC models. The gap between the best and worst models is smaller than on Dataset 2, suggesting the remaining errors are harder to eliminate with these compact model families. k-NN with k=1 overfits markedly (train–test macro-F1 gap 0.238); k=5 and k=11 reduce this gap substantially.

### Overall interpretation
Macro-AUC can exceed accuracy because AUC measures ranking quality over thresholds, whereas accuracy evaluates one hard class decision at the estimator's chosen threshold/argmax. For example, Dataset 3's best forest has test macro-AUC around 0.986 while accuracy is 0.853: class probabilities rank positives above negatives well even though the final multiclass decisions still make errors. The single MLP convergence warning is reported rather than hidden. See the PDF for model-by-model plots and per-class tables.

## 5. Saved figures

The workflow saves three exploratory scatter plots and 45 model-output figures (one per dataset/model). Each model-output figure contains its decision regions with true-coloured test points and train/test one-vs-rest micro-ROC curves. The PDF report includes the full set of model-output figures.